# 贝叶斯分类

## 贝叶斯定理(Bayes)
贝叶斯定理告诉我们如何交换条件概率中的条件和结果
$$
P(H|X) = \frac{P(X|H)P(H)}{P(X)}
$$
P(H|X)给定观测数据样本X，假设H是成立的概率

如X是一分具有特定特征的邮件，H是垃圾邮件。它里面包含很多的单词(特征)，然后我们判断这款邮件属于垃圾邮件的概率

P(H|X)是后验概率。如一份特定邮件，是垃圾邮件的概率

P(H)是H的先验概率。如总体邮件中垃圾邮件的概率

P(X)是X的先验概率。如总体邮件中带有特定特征的邮件概率。

## 朴素贝叶斯(Naive Bayes)
假设：特征X1，X2，X3...之间都是相互独立的
$$
P(H|X) = \frac{P(X|H)P(H)}{P(X)} = \frac{P(X_1|H)P(X_2|H)...P(X_n|H)P(H)}{P(X_1)P(X_2)...P(X_n)}
$$

### 多项式模型
重复的词语视为多次
### 伯努利模型
重复的词语视为一次
### 混合模型
在计算句子概率时，不考虑重复词语出现的次数，但在统计计算词语的概率P("词语"|S)时，考虑重复词语的出现次数
### 高斯模型
有些特征可能是连续变量，可以使用高斯模型进行转换

In [11]:
import numpy as np
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.naive_bayes import MultinomialNB, BernoulliNB, GaussianNB

In [14]:
iris = datasets.load_iris()
x_train, x_test, y_train, y_test = train_test_split(iris.data, iris.target)

In [21]:
# mul_nb = MultinomialNB()
mul_nb = GaussianNB()
mul_nb.fit(x_train, y_train)

GaussianNB(priors=None, var_smoothing=1e-09)

In [22]:
print(classification_report(mul_nb.predict(x_test), y_test))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        13
           1       0.92      1.00      0.96        12
           2       1.00      0.92      0.96        13

    accuracy                           0.97        38
   macro avg       0.97      0.97      0.97        38
weighted avg       0.98      0.97      0.97        38



In [23]:
print(confusion_matrix(mul_nb.predict(x_test), y_test))

[[13  0  0]
 [ 0 12  0]
 [ 0  1 12]]


## 贝叶斯-新闻分类

In [33]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split

In [34]:
news = fetch_20newsgroups(subset='all')
print(news.target_names)

['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']


In [35]:
x_train, x_test, y_train, y_test = train_test_split(news.data, news.target)

### 词袋模型(Bag of Words)
Bag-of-words model(BoW model)最早出现在自然语言处理(Natural Language Processing)和信息检索(Information Retrieval)邻域。该模型忽略掉文本的语法和语序等要素，将其仅仅看作是若干个词汇的集合，文档中每个单词的出现都是独立的。Bow使用一组无序单词来表达一段文字或一个文档

### IF-IDF
提取文章关键词
1. 提取词频(Term Frequency,缩写TF),但需要去掉停用词(stop words)
2. 对关键词排序，将有意义的词排在前面，对每个词分配权重，这个权重叫做"逆文档频率"(Inverse Document Frequency,缩写IDF)，它的大小与一个词的常见程度成反比
$$
词频(TF) = 某个词在文章中出现的次数 \\
词频(TF) = \frac{某个词在文章中的出现次数}{文章的总词数} \\
词频(TF) = \frac{某个词在文章中的出现次数}{该文出现次数最多的词的出现次数} \\
逆文档频率(IDF) = log(\frac{语料库的文档总数}{包含该词的文档数 + 1}) \\
TF-IDF = 词频(TF) * 逆文档频率(IDF)
$$

In [30]:
# CountVectorizer方法构建单词的字典，每个单词实例被转换为特征向量的一个特征数值，每个元素是特定单词在文本中出现的次数
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn import model_selection
from sklearn.naive_bayes import MultinomialNB,BernoulliNB

cv = CountVectorizer()
cv_data = cv.fit_transform(x_train)

mul_nb = MultinomialNB()
score = model_selection.cross_val_score(mul_nb, cv_data, y_train, cv=3, scoring='accuracy')

score.mean()


0.8193715157135036

In [46]:
def get_stop_words():
  result = set()
  for line in open('stopwords_en.txt', 'r').readlines():
    result.add(line.strip())
  return result
stop_words = get_stop_words()
vectorizer = TfidfVectorizer(stop_words=stop_words)
mul_nb = MultinomialNB(alpha=0.01)
tfidf_train = vectorizer.fit_transform(x_train)
score = model_selection.cross_val_score(mul_nb, tfidf_train, y_train, cv=3, scoring='accuracy')
score.mean()

/usr/local/lib/python3.6/dist-packages/sklearn/feature_extraction/text.py:385: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['camq', 're'] not in stop_words.
  'stop_words.' % sorted(inconsistent))


0.9004529730115444

In [47]:
# 切分数据集
tfidf_data = vectorizer.fit_transform(news.data)
x_train,x_test,y_train,y_test = train_test_split(tfidf_data,news.target)

mul_nb.fit(x_train,y_train)
print(mul_nb.score(x_train, y_train))

print(mul_nb.score(x_test, y_test))

0.9959671713598415
0.9178692699490663
